# MOSS-Image-SFT 量子化モデル + 4ステップ LoRA を Kaggle Dataset にアップロードする

Hugging Face の `Comfy-Org/MOSS-Image` から量子化モデル3点（合計約12.1 GiB）と、`Kijai/MOSS-image_experimental` から 4ステップ蒸留 LoRA（約1 GB）をダウンロードし、Kaggle Dataset `masashiki/moss-image-sft-quantized` の**新しいバージョン**としてアップロードします。

## 前提
- **Add-ons → Secrets** に `KAGGLE_API_TOKEN` を登録しておくこと（Kaggle API トークン）

## 注意
- `/kaggle/working` は 20 GB しかないため、ダウンロード先は `/` 直下（`/moss-image-sft-quantized`）にしています
- ディレクトリ構造（`diffusion_models/`, `text_encoders/`, `vae/`, `loras/`）は ComfyUI の `models/` 配置と同じです
- Kaggle CLI はディレクトリをそのままアップロードできないため、`-r tar` でアップロードします（サーバー側で展開され、構造は保持されます）


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    os.environ["KAGGLE_API_TOKEN"] = token
    print("KAGGLE_API_TOKEN: loaded from Secrets")
except Exception as e:
    raise RuntimeError(
        "Secrets に KAGGLE_API_TOKEN が登録されていません。"
        "Add-ons → Secrets から Kaggle API トークンを登録してください。"
    ) from e

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)

In [ ]:
import os

os.environ["TMPDIR"] = "/moss-tmp"
os.makedirs("/moss-tmp", exist_ok=True)

from huggingface_hub import hf_hub_download

files = [
    ("Comfy-Org/MOSS-Image", "diffusion_models/moss_image_sft_int8_convrot.safetensors"),
    ("Comfy-Org/MOSS-Image", "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors"),
    ("Comfy-Org/MOSS-Image", "vae/flux2-vae_small.safetensors"),
    ("Kijai/MOSS-image_experimental", "loras/moss_image_sft_4_steps_lora_v2_600_avg_rank_72_bf16.safetensors"),
]
for repo, fname in files:
    p = hf_hub_download(repo_id=repo, filename=fname,
                        local_dir="/moss-image-sft-quantized")
    print("downloaded:", p)

In [ ]:
import json

metadata = {
    "title": "MOSS-Image-SFT Quantized",
    "id": "masashiki/moss-image-sft-quantized",
    "licenses": [{"name": "apache-2.0"}],
}
with open("/moss-image-sft-quantized/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

In [ ]:
import subprocess

res = subprocess.run(
    ["kaggle", "datasets", "version", "-p", "/moss-image-sft-quantized", "-r", "tar",
     "-m", "add 4-step LoRA (Kijai/MOSS-image_experimental)"],
    capture_output=True,
    text=True,
)
print(res.stdout)
print(res.stderr)

In [ ]:
import subprocess

res = subprocess.run(
    ["kaggle", "datasets", "files", "masashiki/moss-image-sft-quantized"],
    capture_output=True,
    text=True,
)
print(res.stdout)
print(res.stderr)